In [37]:
import sys
!{sys.executable} -m pip install nltk evaluate transformers torch scikit-learn pandas accelerate seqeval

In [38]:
# =========================
# IMPORT REQUIRED LIBRARIES
# =========================

import os
import numpy as np
import pandas as pd
import torch
import nltk
import evaluate

from nltk.corpus import treebank
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [39]:
# ===================================
# DOWNLOAD NLTK DATASET AND TAGGER DATA
# ===================================

nltk.download("treebank")
nltk.download("universal_tagset")

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


True

In [40]:
# =========================
# CHECK DEVICE (CPU / GPU)
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [41]:
# =========================
# LOAD TREEBANK SENTENCES
# =========================

# Get tagged sentences from treebank using universal POS tags
tagged_sentences = treebank.tagged_sents(tagset="universal")

# Print one sample sentence
print("Sample tagged sentence:")
print(tagged_sentences[0])

Sample tagged sentence:
[('Pierre', 'NOUN'), ('Vinken', 'NOUN'), (',', '.'), ('61', 'NUM'), ('years', 'NOUN'), ('old', 'ADJ'), (',', '.'), ('will', 'VERB'), ('join', 'VERB'), ('the', 'DET'), ('board', 'NOUN'), ('as', 'ADP'), ('a', 'DET'), ('nonexecutive', 'ADJ'), ('director', 'NOUN'), ('Nov.', 'NOUN'), ('29', 'NUM'), ('.', '.')]


In [42]:
# ===========================================
# EXTRACT TOKENS AND POS LABELS FROM DATASET
# ===========================================

sentences = []
pos_labels = []

for sent in tagged_sentences:
    words = [word for word, tag in sent]
    tags = [tag for word, tag in sent]
    
    sentences.append(words)
    pos_labels.append(tags)

print("Total sentences:", len(sentences))
print("First sentence:", sentences[0])
print("First POS labels:", pos_labels[0])

Total sentences: 3914
First sentence: ['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']
First POS labels: ['NOUN', 'NOUN', '.', 'NUM', 'NOUN', 'ADJ', '.', 'VERB', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'ADJ', 'NOUN', 'NOUN', 'NUM', '.']


In [43]:
# =====================================================
# CREATE SIMPLE CHUNK LABELS FROM POS TAGS
# =====================================================
# This is a rule-based approximation for chunking
# so that we can complete the chunking task as well.

def generate_chunk_tags(pos_sequence):
    """
    Generate simple chunk tags from POS labels.
    
    Rules:
    - NOUN/ADJ/DET/PRON/NUM -> NP (Noun Phrase)
    - VERB/ADV -> VP (Verb Phrase)
    - ADP -> PP (Prepositional Phrase)
    - Others -> O
    """
    chunk_tags = []
    prev_chunk = "O"

    for pos in pos_sequence:
        if pos in ["NOUN", "ADJ", "DET", "PRON", "NUM"]:
            current = "NP"
        elif pos in ["VERB", "ADV"]:
            current = "VP"
        elif pos == "ADP":
            current = "PP"
        else:
            current = "O"

        if current == "O":
            chunk_tags.append("O")
        else:
            if current != prev_chunk:
                chunk_tags.append("B-" + current)
            else:
                chunk_tags.append("I-" + current)

        prev_chunk = current

    return chunk_tags

# Create chunk labels for all sentences
chunk_labels = [generate_chunk_tags(tags) for tags in pos_labels]

print("First chunk labels:", chunk_labels[0])

First chunk labels: ['B-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'I-NP', 'I-NP', 'O']


In [44]:
# ==========================================
# REDUCE DATASET SIZE FOR FASTER EXECUTION
# ==========================================

# Keep smaller subset for laptop-friendly training
sentences = sentences[:2000]
pos_labels = pos_labels[:2000]
chunk_labels = chunk_labels[:2000]

print("Reduced dataset size:", len(sentences))

Reduced dataset size: 2000


In [45]:
# ====================================
# SPLIT DATA INTO TRAIN / VAL / TEST
# ====================================

from sklearn.model_selection import train_test_split

# First split: train and temp
train_sentences, temp_sentences, train_pos, temp_pos, train_chunk, temp_chunk = train_test_split(
    sentences, pos_labels, chunk_labels, test_size=0.2, random_state=42
)

# Second split: validation and test
val_sentences, test_sentences, val_pos, test_pos, val_chunk, test_chunk = train_test_split(
    temp_sentences, temp_pos, temp_chunk, test_size=0.5, random_state=42
)

print("Train size:", len(train_sentences))
print("Validation size:", len(val_sentences))
print("Test size:", len(test_sentences))

Train size: 1600
Validation size: 200
Test size: 200


In [46]:
# ====================================
# CREATE LABEL MAPPINGS
# ====================================

# Unique POS labels
pos_label_list = sorted(list(set(label for sent in train_pos for label in sent)))

# Unique Chunk labels
chunk_label_list = sorted(list(set(label for sent in train_chunk for label in sent)))

# POS mappings
pos_label2id = {label: i for i, label in enumerate(pos_label_list)}
pos_id2label = {i: label for i, label in enumerate(pos_label_list)}

# Chunk mappings
chunk_label2id = {label: i for i, label in enumerate(chunk_label_list)}
chunk_id2label = {i: label for i, label in enumerate(chunk_label_list)}

print("POS Labels:", pos_label_list)
print("Chunk Labels:", chunk_label_list)

POS Labels: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
Chunk Labels: ['B-NP', 'B-PP', 'B-VP', 'I-NP', 'I-PP', 'I-VP', 'O']


In [47]:
# ====================================
# LOAD DISTILBERT TOKENIZER
# ====================================

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [48]:
# ==========================================================
# CUSTOM DATASET CLASS FOR TOKEN CLASSIFICATION
# ==========================================================

class TokenClassificationDataset(Dataset):
    """
    Custom PyTorch dataset for token classification.
    
    Handles:
    - tokenization
    - label alignment
    - subword handling
    - special token handling (-100)
    """

    def __init__(self, sentences, labels, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        word_labels = self.labels[idx]

        # Tokenize sentence
        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Map token positions back to original word positions
        word_ids = encoding.word_ids(batch_index=0)

        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            # Special tokens ([CLS], [SEP], [PAD]) -> ignore
            if word_idx is None:
                label_ids.append(-100)

            # First token of a word -> assign label
            elif word_idx != previous_word_idx:
                label_ids.append(self.label2id[word_labels[word_idx]])

            # Subword tokens -> ignore
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        # Remove batch dimension
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(label_ids)

        return item

In [49]:
# ====================================
# CREATE POS DATASETS
# ====================================

train_pos_dataset = TokenClassificationDataset(train_sentences, train_pos, tokenizer, pos_label2id)
val_pos_dataset = TokenClassificationDataset(val_sentences, val_pos, tokenizer, pos_label2id)
test_pos_dataset = TokenClassificationDataset(test_sentences, test_pos, tokenizer, pos_label2id)

print("POS datasets created successfully!")

POS datasets created successfully!


In [50]:
# ====================================
# CREATE CHUNK DATASETS
# ====================================

train_chunk_dataset = TokenClassificationDataset(train_sentences, train_chunk, tokenizer, chunk_label2id)
val_chunk_dataset = TokenClassificationDataset(val_sentences, val_chunk, tokenizer, chunk_label2id)
test_chunk_dataset = TokenClassificationDataset(test_sentences, test_chunk, tokenizer, chunk_label2id)

print("Chunk datasets created successfully!")

Chunk datasets created successfully!


In [51]:
# ====================================
# DATA COLLATOR
# ====================================

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [52]:
# ====================================
# LOAD SEQEVAL METRIC
# ====================================

seqeval = evaluate.load("seqeval")

In [53]:
# ====================================
# METRIC FUNCTION FOR POS TAGGING
# ====================================

def compute_metrics_pos(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [pos_label_list[pred] for pred, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [pos_label_list[lab] for pred, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [54]:
# ====================================
# METRIC FUNCTION FOR CHUNKING
# ====================================

def compute_metrics_chunk(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [chunk_label_list[pred] for pred, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [chunk_label_list[lab] for pred, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [55]:
# ====================================
# LOAD MODEL FOR POS TAGGING
# ====================================

pos_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(pos_label_list),
    id2label=pos_id2label,
    label2id=pos_label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [56]:
# ====================================
# TRAINING ARGUMENTS FOR POS TAGGING
# ====================================

pos_training_args = TrainingArguments(
    output_dir="./pos_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,   # keep 1 for faster laptop training
    weight_decay=0.01,
    logging_dir="./logs_pos",
    logging_steps=50,
    save_total_limit=1,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [57]:
# ====================================
# CREATE TRAINER FOR POS TAGGING
# ====================================

pos_trainer = Trainer(
    model=pos_model,
    args=pos_training_args,
    train_dataset=train_pos_dataset,
    eval_dataset=val_pos_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_pos
)

In [58]:
# ====================================
# TRAIN POS MODEL
# ====================================

pos_trainer.train()

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.133952,0.108956,0.952591,0.947925,0.950252,0.969549


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: ADJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: NOUN s

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=400, training_loss=0.4494379258155823, metrics={'train_runtime': 685.6011, 'train_samples_per_second': 2.334, 'train_steps_per_second': 0.583, 'total_flos': 52270689484800.0, 'train_loss': 0.4494379258155823, 'epoch': 1.0})

In [61]:
# ====================================
# SAFE EVALUATION (NO TRAINER STATE ISSUE)
# ====================================

from torch.utils.data import DataLoader

# Put model in evaluation mode
pos_model.eval()

# Create DataLoader manually
eval_loader = DataLoader(val_pos_dataset, batch_size=4)

all_preds = []
all_labels = []

# Disable gradient calculation
with torch.no_grad():
    for batch in eval_loader:
        inputs = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"]
        }

        labels = batch["labels"]

        outputs = pos_model(**inputs)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=2)

        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

# Convert to proper format for seqeval
true_predictions = []
true_labels = []

for pred, label in zip(all_preds, all_labels):
    pred_labels = []
    true_lbls = []

    for p, l in zip(pred, label):
        if l != -100:
            pred_labels.append(pos_id2label[p])
            true_lbls.append(pos_id2label[l])

    true_predictions.append(pred_labels)
    true_labels.append(true_lbls)

# Compute metrics
results = seqeval.compute(predictions=true_predictions, references=true_labels)

print("POS Evaluation Results:")
print(results)

POS Evaluation Results:
{'DJ': {'precision': np.float64(0.8403908794788274), 'recall': np.float64(0.8269230769230769), 'f1': np.float64(0.8336025848142163), 'number': np.int64(312)}, 'DP': {'precision': np.float64(0.9575757575757575), 'recall': np.float64(0.973305954825462), 'f1': np.float64(0.965376782077393), 'number': np.int64(487)}, 'DV': {'precision': np.float64(0.8493975903614458), 'recall': np.float64(0.8245614035087719), 'f1': np.float64(0.8367952522255192), 'number': np.int64(171)}, 'ERB': {'precision': np.float64(0.9642248722316865), 'recall': np.float64(0.948073701842546), 'f1': np.float64(0.9560810810810811), 'number': np.int64(597)}, 'ET': {'precision': np.float64(0.990521327014218), 'recall': np.float64(0.9835294117647059), 'f1': np.float64(0.987012987012987), 'number': np.int64(425)}, 'ONJ': {'precision': np.float64(0.9752066115702479), 'recall': np.float64(0.9915966386554622), 'f1': np.float64(0.9833333333333334), 'number': np.int64(119)}, 'OUN': {'precision': np.float6

In [62]:
# ====================================
# SAVE POS MODEL
# ====================================

pos_model.save_pretrained("./saved_pos_model")
tokenizer.save_pretrained("./saved_pos_model")

print("POS model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

POS model saved successfully!


In [63]:
# ====================================
# LOAD MODEL FOR CHUNKING
# ====================================

chunk_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(chunk_label_list),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [64]:
# ====================================
# TRAINING ARGUMENTS FOR CHUNKING
# ====================================

chunk_training_args = TrainingArguments(
    output_dir="./chunk_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs_chunk",
    logging_steps=50,
    save_total_limit=1,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [66]:
# ====================================
# CREATE TRAINER FOR CHUNKING
# ====================================

chunk_trainer = Trainer(
    model=chunk_model,
    args=chunk_training_args,
    train_dataset=train_chunk_dataset,
    eval_dataset=val_chunk_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_chunk
)

print("Chunk Trainer created successfully!")

Chunk Trainer created successfully!


In [67]:
# ====================================
# TRAIN CHUNK MODEL
# ====================================

chunk_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.133739,0.117906,0.928222,0.933935,0.931070,0.968573


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=400, training_loss=0.3617553734779358, metrics={'train_runtime': 660.398, 'train_samples_per_second': 2.423, 'train_steps_per_second': 0.606, 'total_flos': 52265964748800.0, 'train_loss': 0.3617553734779358, 'epoch': 1.0})

In [69]:
# ====================================
# SAFE EVALUATION FOR CHUNK MODEL
# ====================================

from torch.utils.data import DataLoader

# Put model in evaluation mode
chunk_model.eval()

# Create DataLoader manually
eval_loader = DataLoader(val_chunk_dataset, batch_size=4)

all_preds = []
all_labels = []

# Disable gradient calculation
with torch.no_grad():
    for batch in eval_loader:
        inputs = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"]
        }

        labels = batch["labels"]

        outputs = chunk_model(**inputs)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=2)

        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

# Convert predictions and labels into readable format
true_predictions = []
true_labels = []

for pred, label in zip(all_preds, all_labels):
    pred_labels = []
    true_lbls = []

    for p, l in zip(pred, label):
        if l != -100:
            pred_labels.append(chunk_id2label[p])
            true_lbls.append(chunk_id2label[l])

    true_predictions.append(pred_labels)
    true_labels.append(true_lbls)

# Compute metrics
chunk_eval_results = seqeval.compute(predictions=true_predictions, references=true_labels)

print("Chunk Evaluation Results:")
print(chunk_eval_results)

Chunk Evaluation Results:
{'NP': {'precision': np.float64(0.9472477064220184), 'recall': np.float64(0.9436405178979437), 'f1': np.float64(0.9454406714994278), 'number': np.int64(1313)}, 'PP': {'precision': np.float64(0.9442231075697212), 'recall': np.float64(0.973305954825462), 'f1': np.float64(0.9585439838220424), 'number': np.int64(487)}, 'VP': {'precision': np.float64(0.8769470404984424), 'recall': np.float64(0.8838304552590267), 'f1': np.float64(0.8803752931978108), 'number': np.int64(637)}, 'overall_precision': np.float64(0.9282218597063622), 'overall_recall': np.float64(0.933935166187936), 'overall_f1': np.float64(0.9310697484148088), 'overall_accuracy': 0.9685731016982237}


In [70]:
# ====================================
# SAVE CHUNK MODEL
# ====================================

chunk_model.save_pretrained("./saved_chunk_model")
tokenizer.save_pretrained("./saved_chunk_model")

print("Chunk model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk model saved successfully!


In [71]:
# ====================================
# POS INFERENCE FUNCTION
# ====================================

def predict_pos(sentence):
    """
    Predict POS tags for a custom sentence.
    """
    words = sentence.split()

    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    pos_model.to(device)

    with torch.no_grad():
        outputs = pos_model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()
    word_ids = inputs.encodings[0].word_ids

    predicted_labels = []
    seen_word_ids = set()

    for pred, word_id in zip(predictions, word_ids):
        if word_id is not None and word_id not in seen_word_ids:
            predicted_labels.append(pos_id2label[pred])
            seen_word_ids.add(word_id)

    print("POS Predictions:")
    for word, label in zip(words, predicted_labels):
        print(f"{word:15} -> {label}")

In [75]:
# ====================================
# POS INFERENCE FUNCTION
# ====================================

def predict_pos(sentence):
    """
    Predict POS tags for a custom sentence.
    """

    # Split sentence into words
    words = sentence.split()

    # Tokenize as split words
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Put model in evaluation mode
    pos_model.eval()

    with torch.no_grad():
        outputs = pos_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

    # Get predicted class IDs
    predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()

    # Correct way to get word IDs
    word_ids = inputs.word_ids(batch_index=0)

    predicted_labels = []
    seen_word_ids = set()

    for pred, word_id in zip(predictions, word_ids):
        if word_id is not None and word_id not in seen_word_ids:
            predicted_labels.append(pos_id2label[pred])
            seen_word_ids.add(word_id)

    print("\nPOS Predictions:")
    for word, label in zip(words, predicted_labels):
        print(f"{word:15} -> {label}")

In [80]:
# ====================================
# CHUNK INFERENCE FUNCTION
# ====================================

def predict_chunk(sentence):
    """
    Predict chunk tags for a custom sentence.
    """

    # Split sentence into words
    words = sentence.split()

    # Tokenize as split words
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Put model in evaluation mode
    chunk_model.eval()

    with torch.no_grad():
        outputs = chunk_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

    # Get predicted class IDs
    predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()

    # Correct way to get word IDs
    word_ids = inputs.word_ids(batch_index=0)

    predicted_labels = []
    seen_word_ids = set()

    for pred, word_id in zip(predictions, word_ids):
        if word_id is not None and word_id not in seen_word_ids:
            predicted_labels.append(chunk_id2label[pred])
            seen_word_ids.add(word_id)

    print("\nChunk Predictions:")
    for word, label in zip(words, predicted_labels):
        print(f"{word:15} -> {label}")

In [81]:
# Test Chunk model
predict_chunk("John works at Google in California")


Chunk Predictions:
John            -> B-NP
works           -> B-VP
at              -> B-PP
Google          -> B-NP
in              -> B-PP
California      -> B-NP


In [84]:
# ====================================
# CREATE FINAL RESULT SUMMARY TABLE
# ====================================

summary_df = pd.DataFrame({
    "Task": ["POS Tagging", "Chunking"],
    "Precision": [results["overall_precision"], chunk_eval_results["overall_precision"]],
    "Recall": [results["overall_recall"], chunk_eval_results["overall_recall"]],
    "F1 Score": [results["overall_f1"], chunk_eval_results["overall_f1"]],
    "Accuracy": [results["overall_accuracy"], chunk_eval_results["overall_accuracy"]],
})

summary_df

,Task,Precision,Recall,F1 Score,Accuracy
0,POS Tagging,0.952591,0.947925,0.950252,0.969549
1,Chunking,0.928222,0.933935,0.931070,0.968573


In [85]:
# ====================================
# SAVE FINAL RESULTS TO CSV
# ====================================

os.makedirs("outputs", exist_ok=True)
summary_df.to_csv("outputs/evaluation_summary.csv", index=False)

print("Evaluation summary saved successfully!")

Evaluation summary saved successfully!


Observations
DistilBERT performed well on both token classification tasks.
POS tagging was comparatively easier because it works at the word level.
Chunking was slightly more difficult because it requires identifying phrase boundaries.
Label alignment was one of the most important preprocessing steps.
Challenges Faced
Handling subword tokenization
Aligning labels correctly with tokens
Ignoring special tokens using -100
Training transformer models efficiently on a laptop